In [ ]:
import pandas as pd
import numpy as np
data = pd.read_csv('data/processed/final_dataset_fe.csv')

In [52]:
display(data.head())

,daynight_N,lat,lon,fire_weather_index,pressure_mean,wind_direction_std,solar_radiation_mean,dewpoint_mean,cloud_cover_mean,evapotranspiration_total,...,temp_range,wind_speed_max,frp,frp_log,wind_direction_sin,wind_direction_cos,wind_speed_max_log,fire_weather_index_log,hot_dry_index,wind_dry_index
0,0.0,-15.19928,38.54393,5.654271,955.608333,43.611845,250.333333,15.883333,18.375000,4.89,...,12.9,13.5,3.69,1.545433,0.693611,-0.720349,2.674149,1.895259,0.643750,0.375000
1,1.0,31.51203,-101.57546,16.564673,927.016667,31.925260,296.916667,10.529167,3.416667,7.24,...,13.4,16.2,0.73,0.548121,-0.023269,-0.999729,2.844909,2.865890,1.749020,0.952941
2,1.0,-13.74538,28.05493,5.542089,884.379167,8.379870,210.958333,6.095833,12.208333,3.68,...,11.1,16.8,0.78,0.576613,0.944568,-0.328317,2.879198,1.878256,0.391782,0.466667
3,0.0,13.53091,-11.22573,7.703410,981.158333,133.287570,244.208333,13.095833,56.458333,5.59,...,12.9,11.0,8.91,2.293544,0.200793,-0.979634,2.484907,2.163715,0.890139,0.366667
4,0.0,5.61833,16.59892,16.895093,917.116667,91.791875,266.750000,6.712500,9.166667,6.00,...,16.0,16.6,5.97,1.941615,-0.377975,-0.925816,2.867899,2.884527,2.438258,1.509091


In [53]:
data.shape

(118858, 22)

## 1. Постановка задачи и выбор признаков

Целевая переменная — `frp`. В обновлённом датасете после feature engineering уже есть `frp_log`, поэтому отдельно создавать логарифм таргета не нужно: используем `frp` для метрик в исходной шкале, а `frp_log` — как таргет для обучения моделей. В признаки не включаем ни `frp`, ни `frp_log`, потому что это целевые значения.

In [54]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import wandb
%matplotlib inline
import copy
import joblib
from pathlib import Path

In [55]:
#Функция для установки всех сидов для воспроизводимости результатов.
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

#Конфиг с семинаров
class CFG:
    seed = 42
    num_epochs = 25
    train_batch_size = 256
    test_batch_size = 1024
    lr = 0.001
    wandb = True
    project = 'gp5-wildfire-frp'
    entity = None 


seed_everything(CFG.seed)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cpu')

In [56]:
#Сюда будем сохранять модели
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)


### Настройка Weights & Biases

Логирование экспериментов сделано по образцу Seminar 15 - Neural Networks.ipynb: в семинаре используются `wandb.init(...)`, `wandb.watch(...)` и `wandb.log(...)`. В этом проекте каждый baseline/MLP запуск логируется отдельным run. API-ключ не записываем в ноутбук; его нужно ввести локально через `wandb login` или `wandb.login()`.

In [57]:
def start_wandb_run(run_name, config):
    return wandb.init(
        project=CFG.project,
        entity=CFG.entity,
        name=run_name,
        reinit=True,
        config=config
    )


def log_wandb(metrics):
    wandb.log(metrics)


def finish_wandb_run():
    wandb.finish()

In [ ]:
wandb.login(relogin=True)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\admin\_netrc
wandb: Currently logged in as: creativedak567 (creativedak567-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [58]:
y_log = data['frp_log'].copy()
data.drop(columns=['frp', 'frp_log'], inplace=True)
X = data.copy()

print('Количество признаков:', len(X.columns))
print('Признаки модели:')
print(X.columns)

Количество признаков: 20
Признаки модели:
Index(['daynight_N', 'lat', 'lon', 'fire_weather_index', 'pressure_mean',
       'wind_direction_std', 'solar_radiation_mean', 'dewpoint_mean',
       'cloud_cover_mean', 'evapotranspiration_total', 'humidity_min',
       'temp_mean', 'temp_range', 'wind_speed_max', 'wind_direction_sin',
       'wind_direction_cos', 'wind_speed_max_log', 'fire_weather_index_log',
       'hot_dry_index', 'wind_dry_index'],
      dtype='str')


## 2. Train / validation / test и подготовка данных

Для регрессии используем `log1p(frp)`, потому что `frp` обычно имеет скошенное распределение и выбросы. Модель обучается в логарифмической шкале, а метрики считаются после обратного преобразования через `expm1`.

In [59]:
# Делаем стратификацию по бинам frp_log, чтобы train/val/test были похожи по распределению интенсивности.
stratify_bins = pd.qcut(y_log, q=10, labels=False, duplicates='drop')

X_train_val, X_test, y_train_val_log, y_test_log_series = train_test_split(
    X, y_log,
    test_size=0.15,
    random_state=CFG.seed,
    stratify=stratify_bins
)

train_val_bins = pd.qcut(y_train_val_log, q=10, labels=False, duplicates='drop')

X_train, X_val, y_train_log_series, y_val_log_series = train_test_split(
    X_train_val, y_train_val_log,
    test_size=0.1765,
    random_state=CFG.seed,
    stratify=train_val_bins
)

print('Train:', X_train.shape, y_train_log_series.shape)
print('Validation:', X_val.shape, y_val_log_series.shape)
print('Test:', X_test.shape, y_test_log_series.shape)

Train: (83197, 20) (83197,)
Validation: (17832, 20) (17832,)
Test: (17829, 20) (17829,)


In [60]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

y_train_log = y_train_log_series.values.astype('float32')
y_val_log = y_val_log_series.values.astype('float32')
y_test_log = y_test_log_series.values.astype('float32')

X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train_log).view(-1, 1)

X_val_tensor = torch.FloatTensor(X_val_scaled)
y_val_tensor = torch.FloatTensor(y_val_log).view(-1, 1)

X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test_log).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=CFG.train_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CFG.test_batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CFG.test_batch_size, shuffle=False)

Выделим основные метрики для задачи регрессии

In [61]:
def regression_metrics(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred)
    }

## 3. Baseline без DL

Перед MLP обучаем простую табличную модель, чтобы понимать, есть ли польза от нейросетей относительно классического baseline.

In [62]:
baseline = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    random_state=CFG.seed,
    n_jobs=-1
)

wandb.init(
    project=CFG.project,
    entity=CFG.entity,
    name='RandomForest baseline',
    reinit=True,
    config={
        'model': 'RandomForestRegressor',
        'n_estimators': 100,
        'max_depth': 12,
        'target_transform': 'log1p(frp)'
    }
)

baseline.fit(X_train_scaled, y_train_log)

baseline_model_path = MODEL_DIR / 'random_forest_baseline_best.joblib'
joblib.dump(baseline, baseline_model_path)

baseline_val_pred = np.expm1(baseline.predict(X_val_scaled))
baseline_test_pred = np.expm1(baseline.predict(X_test_scaled))


y_test_baseline_true = np.expm1(y_test_log_series.values)
baseline_metrics = regression_metrics(y_test_baseline_true, baseline_test_pred)
if CFG.wandb and wandb is not None:
    wandb.log({f'test_{key}': value for key, value in baseline_metrics.items()})
    wandb.save(str(baseline_model_path))
    wandb.finish()

baseline_metrics

wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


test_MAE,▁
test_R2,▁
test_RMSE,▁
test_MAE,12.57227
test_R2,0.06551
test_RMSE,31.66309


{'MAE': 12.572274225340118,
 'RMSE': np.float64(31.6630882534637),
 'R2': 0.06551188930978535}

## 4. MLP-модели

Дальше используем конструкции из семинаров 15 и 18-19: `nn.Module`, `nn.Sequential`, `Linear`, `ReLU`, `BatchNorm1d`, `Dropout`, `MSELoss`, `Adam`, циклы `train/eval`.

Начнем обучение с обычной MLPRegressionNet

В качестве функции активации возьмем Relu, в качетсве оптимизатора возьмем adam, делать dropout и нормализацию батчей пока что не будем

Создадим класс в общем виде(не будем прописывать количество слоев и нейронов сразу), чтобы в последствии запустить с разным количеством слоев и нейронов

In [63]:
class MLPRegressionNet(nn.Module):
    def __init__(self, input_size, hidden_layers, dropout=0.0, batch_norm=False, activation='linear'):
        super(MLPRegressionNet, self).__init__()
        layers = []
        previous_size = input_size

        for hidden_size in hidden_layers:
            layers.append(nn.Linear(previous_size, hidden_size))

            if batch_norm:
                layers.append(nn.BatchNorm1d(hidden_size))

            if activation == 'relu':
                layers.append(nn.ReLU())
            elif activation == 'tanh':
                layers.append(nn.Tanh())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            previous_size = hidden_size

        layers.append(nn.Linear(previous_size, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)
    
#Функция для обучения одной эпохи и подсчета функции потерь на обучающей выборке
def train_epoch(model, device, train_loader, optimizer, criterion):
    model.train()
    train_loss = 0

    for data_batch, target_batch in train_loader:
        data_batch = data_batch.to(device)
        target_batch = target_batch.to(device)

        optimizer.zero_grad()
        output = model(data_batch)
        loss = criterion(output, target_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * data_batch.size(0)

    return train_loss / len(train_loader.dataset)

#Функция для оценки модели на тестовой выборке для одной эпохи
def test_epoch(model, device, loader, criterion):
    model.eval()
    test_loss = 0
    predictions = []
    targets = []

    with torch.no_grad():
        for data_batch, target_batch in loader:
            data_batch = data_batch.to(device)
            target_batch = target_batch.to(device)

            output = model(data_batch)
            loss = criterion(output, target_batch)
            test_loss += loss.item() * data_batch.size(0)

            predictions.append(output.cpu().numpy())
            targets.append(target_batch.cpu().numpy())

    predictions = np.vstack(predictions).ravel()
    targets = np.vstack(targets).ravel()

    return test_loss / len(loader.dataset), predictions, targets

Создадим функцию для обучения модели чтобы прогонять ее с разными гиперпараметрами

Обучим саму модель, в качестве функции потерь используем MSELoss

In [64]:
def train_mlp_experiment(run_name, hidden_layers, dropout=0.0, batch_norm=False, activation='linear', lr=CFG.lr, weight_decay=0.0, loss_name='MSELoss', num_epochs=CFG.num_epochs):
    seed_everything(CFG.seed)

    model = MLPRegressionNet(
        input_size=X_train_scaled.shape[1],
        hidden_layers=hidden_layers,
        dropout=dropout,
        batch_norm=batch_norm,
        activation=activation
    ).to(DEVICE)

    if loss_name == 'SmoothL1Loss':
        criterion = nn.SmoothL1Loss()
    else:
        criterion = nn.MSELoss()

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    safe_run_name = run_name.lower().replace(' ', '_').replace('/', '_')
    model_path = MODEL_DIR / f'{safe_run_name}_best.pt'

    wandb.init(
        project=CFG.project,
        entity=CFG.entity,
        name=run_name,
        reinit=True,
        config={
            'model': run_name,
            'hidden_layers': hidden_layers,
            'dropout': dropout,
            'batch_norm': batch_norm,
            'activation': activation,
            'lr': lr,
            'weight_decay': weight_decay,
            'loss': loss_name,
            'target_transform': 'frp_log',
            'epochs': num_epochs,
            'batch_size': CFG.train_batch_size,
            'model_path': str(model_path)
        }
    )
    wandb.watch(model, log='all')

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = np.inf
    best_epoch = 0
    best_model_state = copy.deepcopy(model.state_dict())

    for epoch in tqdm(range(1, num_epochs + 1), desc=run_name):
        train_loss = train_epoch(model, DEVICE, train_loader, optimizer, criterion)
        val_loss, _, _ = test_epoch(model, DEVICE, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save({
                'model_state_dict': best_model_state,
                'input_size': X_train_scaled.shape[1],
                'hidden_layers': hidden_layers,
                'dropout': dropout,
                'batch_norm': batch_norm,
                'activation': activation,
                'best_epoch': best_epoch,
                'best_val_loss': best_val_loss,
                'target_transform': 'log1p(frp)'
            }, model_path)

        wandb.log({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'best_val_loss': best_val_loss,
            'best_epoch': best_epoch
        })

        if epoch % 5 == 0:
            print(f'Epoch: {epoch}, train_loss: {train_loss:.4f}, val_loss: {val_loss:.4f}, best_val_loss: {best_val_loss:.4f}')

    model.load_state_dict(best_model_state)

    test_loss, test_pred_log, test_target_log = test_epoch(model, DEVICE, test_loader, criterion)
    test_pred = np.expm1(test_pred_log)
    test_target = np.expm1(test_target_log)

    metrics = regression_metrics(test_target, test_pred)
    metrics['test_loss_log_scale'] = test_loss
    metrics['best_epoch'] = best_epoch
    metrics['best_val_loss'] = best_val_loss

    wandb.log({f'test_{key}': value for key, value in metrics.items()})
    wandb.save(str(model_path))
    wandb.finish()

    print(f'Best model saved to: {model_path}')
    print(f'Best epoch: {best_epoch}, best_val_loss: {best_val_loss:.4f}')

    return model, history, metrics


Попробуем обучить модель с словями 64 и 32 без нормализации батчей, дропаута и регуляризации

In [65]:
model_1, history_1, metrics_1 = train_mlp_experiment(
    run_name='MLPRegressionNet 1st Try',
    hidden_layers=[64, 32],
    dropout=0.0,
    batch_norm=False,
    activation='relu',
    lr=CFG.lr,
    weight_decay=0.0,
    loss_name='MSELoss',
    num_epochs=CFG.num_epochs
)

metrics_1

MLPRegressionNet 1st Try:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch: 5, train_loss: 0.6947, val_loss: 0.6938, best_val_loss: 0.6927
Epoch: 10, train_loss: 0.6859, val_loss: 0.6862, best_val_loss: 0.6862
Epoch: 15, train_loss: 0.6805, val_loss: 0.6879, best_val_loss: 0.6833
Epoch: 20, train_loss: 0.6763, val_loss: 0.6797, best_val_loss: 0.6797


wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Epoch: 25, train_loss: 0.6727, val_loss: 0.6787, best_val_loss: 0.6787


best_epoch,▁▁▂▂▂▂▂▃▃▄▄▄▄▄▄▅▅▆▆▇▇▇▇▇█
best_val_loss,█▅▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
test_MAE,▁
test_R2,▁
test_RMSE,▁
test_best_epoch,▁
test_best_val_loss,▁
test_test_loss_log_scale,▁
train_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+1,...


Best model saved to: models\mlpregressionnet_1st_try_best.pt
Best epoch: 25, best_val_loss: 0.6787


{'MAE': 12.733437538146973,
 'RMSE': np.float64(32.030725986567376),
 'R2': 0.0436854362487793,
 'test_loss_log_scale': 0.6792654033142891,
 'best_epoch': 25,
 'best_val_loss': 0.6787027535335854}

По метрикам получаем картину хуже относительно baseline модели, попробуем поменять архитектуру

Сделаем 4 слоя: [256, 128, 64, 32], проверим помогает ли увеличение глубины сети лучше ловить сложные нелинейные связи между данными и frp

In [66]:
model_2, history_2, metrics_2 = train_mlp_experiment(
    run_name='MLPRegressionNet 2nd Try',
    hidden_layers=[256, 128, 64, 32],
    dropout=0.0,
    batch_norm=False,
    activation='relu',
    lr=CFG.lr,
    weight_decay=0.0,
    loss_name='MSELoss',
    num_epochs=CFG.num_epochs
)

metrics_2

MLPRegressionNet 2nd Try:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch: 5, train_loss: 0.6897, val_loss: 0.6865, best_val_loss: 0.6865
Epoch: 10, train_loss: 0.6783, val_loss: 0.6811, best_val_loss: 0.6781
Epoch: 15, train_loss: 0.6690, val_loss: 0.6769, best_val_loss: 0.6751
Epoch: 20, train_loss: 0.6627, val_loss: 0.6807, best_val_loss: 0.6734


wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Epoch: 25, train_loss: 0.6556, val_loss: 0.6728, best_val_loss: 0.6728


best_epoch,▁▁▂▂▂▂▃▃▃▃▄▄▅▅▅▅▅▅▅▅▅▅▅▅█
best_val_loss,█▆▅▅▄▄▄▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
test_MAE,▁
test_R2,▁
test_RMSE,▁
test_best_epoch,▁
test_best_val_loss,▁
test_test_loss_log_scale,▁
train_loss,█▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+1,...


Best model saved to: models\mlpregressionnet_2nd_try_best.pt
Best epoch: 25, best_val_loss: 0.6728


{'MAE': 12.738304138183594,
 'RMSE': np.float64(31.787868234509393),
 'R2': 0.05813199281692505,
 'test_loss_log_scale': 0.6758954112451793,
 'best_epoch': 25,
 'best_val_loss': 0.6727678010740746}

Качество модели не улучшилось, значит уходить в глубину не очень помогает.

Попробуем пойти в ширину и сделать слои [512, 256]

In [67]:
model_3, history_3, metrics_3 = train_mlp_experiment(
    run_name='MLPRegressionNet 3rd Try',
    hidden_layers=[512, 256],
    dropout=0.0,
    batch_norm=False,
    activation='relu',
    lr=CFG.lr,
    weight_decay=0.0,
    loss_name='MSELoss',
    num_epochs=CFG.num_epochs
)

metrics_3

MLPRegressionNet 3rd Try:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch: 5, train_loss: 0.6868, val_loss: 0.6890, best_val_loss: 0.6818
Epoch: 10, train_loss: 0.6748, val_loss: 0.6761, best_val_loss: 0.6761
Epoch: 15, train_loss: 0.6641, val_loss: 0.6849, best_val_loss: 0.6761
Epoch: 20, train_loss: 0.6544, val_loss: 0.6853, best_val_loss: 0.6736


wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Epoch: 25, train_loss: 0.6456, val_loss: 0.6756, best_val_loss: 0.6713


best_epoch,▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▄▄▄▇▇▇▇███
best_val_loss,█▇▆▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
test_MAE,▁
test_R2,▁
test_RMSE,▁
test_best_epoch,▁
test_best_val_loss,▁
test_test_loss_log_scale,▁
train_loss,█▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
+1,...


Best model saved to: models\mlpregressionnet_3rd_try_best.pt
Best epoch: 23, best_val_loss: 0.6713


{'MAE': 12.649309158325195,
 'RMSE': np.float64(32.037518441518294),
 'R2': 0.04327970743179321,
 'test_loss_log_scale': 0.6717088227184043,
 'best_epoch': 23,
 'best_val_loss': 0.6713352264872363}

Качество только ухудшилось, значит сказать что в задаче важнее богатое представление признаков на первых шагах чем глубина сети мы не можем

Конечно мы не будем на этом останавливаться и добавим нормализацию батчей, мы уже заложили возможность этого когда создавали класс нашей модели, так что здесь просто включим ее и посмотрим на результаты, по слоям попробуем взять что то среднее между глубиной и шириной [256, 128, 64]

In [68]:
model_bn, history_bn, metrics_bn = train_mlp_experiment(
    run_name='MLP BatchNorm',
    hidden_layers=[256, 128, 64],
    dropout=0.0,
    batch_norm=True,
    activation='relu',
    lr=CFG.lr,
    weight_decay=0.0,
    loss_name='MSELoss',
    num_epochs=CFG.num_epochs
)

metrics_bn

MLP BatchNorm:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch: 5, train_loss: 0.6765, val_loss: 0.6841, best_val_loss: 0.6816
Epoch: 10, train_loss: 0.6619, val_loss: 0.6749, best_val_loss: 0.6746
Epoch: 15, train_loss: 0.6524, val_loss: 0.6790, best_val_loss: 0.6746
Epoch: 20, train_loss: 0.6406, val_loss: 0.6733, best_val_loss: 0.6716


wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Epoch: 25, train_loss: 0.6298, val_loss: 0.6775, best_val_loss: 0.6716


best_epoch,▁▁▂▂▂▃▄▄▄▄▄▄▄▄▄██████████
best_val_loss,█▅▄▄▄▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
test_MAE,▁
test_R2,▁
test_RMSE,▁
test_best_epoch,▁
test_best_val_loss,▁
test_test_loss_log_scale,▁
train_loss,█▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+1,...


Best model saved to: models\mlp_batchnorm_best.pt
Best epoch: 16, best_val_loss: 0.6716


{'MAE': 12.722381591796875,
 'RMSE': np.float64(31.762963719681647),
 'R2': 0.05960726737976074,
 'test_loss_log_scale': 0.677319327003991,
 'best_epoch': 16,
 'best_val_loss': 0.6715548303748738}

BatchNorm не дал улучшения качества. Метрики почти совпали с baseline, но по MAE модель стала чуть хуже. При этом train loss снижался, а validation loss держался примерно на одном уровне, значит BatchNorm немного стабилизировал обучение, но не улучшил обобщающую способность модели(

Теперь попробуем добавить сюда регуляризацию weight_decay=1e-4, dropout=0.2

И еще попробуем поменять функцию ошибки на SmoothL1Loss, потому что frp обычно имеет выбросы, а MSE сильно на них реагирует

In [69]:
model_bn_reg, history_bn_reg, metrics_bn_reg = train_mlp_experiment(
    run_name='MLP ReLU BatchNorm Dropout SmoothL1',
    hidden_layers=[256, 128, 64],
    dropout=0.2,
    batch_norm=True,
    activation='relu',
    lr=0.001,
    weight_decay=1e-4,
    loss_name='SmoothL1Loss',
    num_epochs=CFG.num_epochs
)

metrics_bn_reg

MLP ReLU BatchNorm Dropout SmoothL1:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch: 5, train_loss: 0.3169, val_loss: 0.2983, best_val_loss: 0.2978
Epoch: 10, train_loss: 0.3087, val_loss: 0.2945, best_val_loss: 0.2945
Epoch: 15, train_loss: 0.3037, val_loss: 0.2955, best_val_loss: 0.2936
Epoch: 20, train_loss: 0.3011, val_loss: 0.2933, best_val_loss: 0.2933


wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Epoch: 25, train_loss: 0.3004, val_loss: 0.2928, best_val_loss: 0.2927


best_epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▆▆▆▆▆▆██████
best_val_loss,██▇▄▄▄▄▄▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
test_MAE,▁
test_R2,▁
test_RMSE,▁
test_best_epoch,▁
test_best_val_loss,▁
test_test_loss_log_scale,▁
train_loss,█▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+1,...


Best model saved to: models\mlp_relu_batchnorm_dropout_smoothl1_best.pt
Best epoch: 21, best_val_loss: 0.2927


{'MAE': 12.61999225616455,
 'RMSE': np.float64(32.3154635518402),
 'R2': 0.0266074538230896,
 'test_loss_log_scale': 0.29403953446870673,
 'best_epoch': 21,
 'best_val_loss': 0.2927235050044182}

Новая модель с SmoothL1Loss, Dropout и weight_decay немного улучшила MAE относительно BatchNorm MLP стало 12.627, а было 12.765, но стала хуже по RMSE и R2. В целом это логично, потому что SmoothL1Loss менее чувствителен к выбросам, поэтому модель лучше работает по средней абсолютной ошибке, но хуже предсказывает редкие большие значения frp, из-за чего растёт RMSE и падает R2

Теперь попробуем убрать регуляризацию, потому что может быть она и ухудшила RMSE и R2

In [70]:
model_smooth_only, history_smooth_only, metrics_smooth_only = train_mlp_experiment(
    run_name='MLP ReLU BatchNorm SmoothL1 only',
    hidden_layers=[256, 128, 64],
    dropout=0.0,
    batch_norm=True,
    activation='relu',
    lr=0.001,
    weight_decay=0.0,
    loss_name='SmoothL1Loss',
    num_epochs=CFG.num_epochs
)

metrics_smooth_only

MLP ReLU BatchNorm SmoothL1 only:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch: 5, train_loss: 0.2947, val_loss: 0.2957, best_val_loss: 0.2951
Epoch: 10, train_loss: 0.2897, val_loss: 0.2932, best_val_loss: 0.2932
Epoch: 15, train_loss: 0.2861, val_loss: 0.2944, best_val_loss: 0.2932
Epoch: 20, train_loss: 0.2816, val_loss: 0.2929, best_val_loss: 0.2929


wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Epoch: 25, train_loss: 0.2781, val_loss: 0.2954, best_val_loss: 0.2929


best_epoch,▁▁▁▂▂▃▃▃▃▄▄▄▄▄▄▇▇▇▇██████
best_val_loss,█▅▅▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
test_MAE,▁
test_R2,▁
test_RMSE,▁
test_best_epoch,▁
test_best_val_loss,▁
test_test_loss_log_scale,▁
train_loss,█▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+1,...


Best model saved to: models\mlp_relu_batchnorm_smoothl1_only_best.pt
Best epoch: 20, best_val_loss: 0.2929


{'MAE': 12.678091049194336,
 'RMSE': np.float64(31.988101654536138),
 'R2': 0.046228885650634766,
 'test_loss_log_scale': 0.29516744018904834,
 'best_epoch': 20,
 'best_val_loss': 0.292893597434165}

RMSE и R2 улучшились по сравнению с вариантом где был Dropout и weight_decay, но всё ещё хуже baseline. Значит проблема была не только в регуляризации. SmoothL1Loss делает модель устойчивее к выбросам, но для нашей задачи это не даёт явного выигрыша, потому что большие значения frp важны для качества по RMSE/R2.

#### Вывод по MLP

После нескольких MLP-экспериментов качество оставалось близким к baseline RandomForest. Обычная полносвязная сеть недостаточно эффективно работает с табличными признаками данного датасета

Следующим шагом была выбрана специализированная нейросетевая архитектура для табличных данных - TabNet, которая использует attention-механизм для отбора информативных признаков и потенциально лучше подходит для нашего датасета

При написании кода мы ориентировались на этот гитхаб репозиторий https://github.com/dreamquark-ai/tabnet в котором описаны основные принципы работы с TabNet и часть кода и некоторых конструкций была взята оттуда

## 5. TabNet для регрессии FRP

### Механизм TabNet

Обычная MLP обрабатывает все признаки сразу и одинаковым способом: на вход подается весь вектор признаков, дальше он проходит через полносвязные слои, и модель постепенно строит прогноз. Такая архитектура универсальна, но для табличных данных она не всегда хорошо понимает, какие признаки действительно важны.

TabNet устроен иначе. Он несколько раз смотрит на признаки и на каждом шаге с помощью attention-механизма выбирает, на какие признаки обратить больше внимания. Поэтому TabNet лучше подходит именно для табличных данных: модель не просто смешивает все признаки в полносвязных слоях, а учится выбирать наиболее полезные признаки для прогноза.

В первом запуске TabNet мы явно задаем следующие гиперпараметры.

- `n_d` - размерность блока, который отвечает за построение прогноза на каждом шаге TabNet. Чем больше значение, тем сложнее модель и тем больше информации она может хранить внутри себя.

- `n_a` - размерность attention-блока, который выбирает важные признаки. Чем больше значение, тем сложнее механизм выбора признаков.

- `n_steps` - количество шагов, на которых TabNet последовательно выбирает признаки и уточняет прогноз. В отличие от MLP, TabNet может несколько раз возвращаться к данным и смотреть на них с разных сторон.

- `gamma` - параметр, который влияет на то, насколько свободно модель может повторно использовать одни и те же признаки на разных шагах. Если значение выше, TabNet чаще может возвращаться к уже использованным признакам.

- `lambda_sparse` - коэффициент разреженности. Он помогает модели выбирать меньше признаков на каждом шаге, а не использовать все сразу. Это делает модель более интерпретируемой и может снижать переобучение.

- `lr` - learning rate, то есть скорость обучения. Он определяет, насколько сильно модель обновляет свои веса после очередного шага обучения.

- `max_epochs` - максимальное количество эпох обучения. Это верхняя граница: модель может остановиться раньше, если validation-качество перестанет улучшаться.

- `patience` - число эпох для early stopping. Если качество на validation set не улучшается заданное количество эпох подряд, обучение останавливается.

- `batch_size` - количество объектов, которое модель обрабатывает за один шаг обучения.

- `virtual_batch_size` - размер виртуального batch внутри TabNet. Он используется для более стабильной нормализации при обучении на больших batch.


In [71]:
from pytorch_tabnet.tab_model import TabNetRegressor

In [72]:
# fit принимает numpy-массивы, а target для регрессии передаем как двумерный массив.

X_train_tabnet = X_train_scaled.astype(np.float32)
X_val_tabnet = X_val_scaled.astype(np.float32)
X_test_tabnet = X_test_scaled.astype(np.float32)

y_train_tabnet = y_train_log_series.values.reshape(-1, 1).astype(np.float32)
y_val_tabnet = y_val_log_series.values.reshape(-1, 1).astype(np.float32)
y_test_tabnet_log = y_test_log_series.values.astype(np.float32)


In [73]:
def train_tabnet_experiment(run_name, n_d=16, n_a=16, n_steps=4, gamma=1.3, lambda_sparse=1e-4, lr=2e-2, max_epochs=100, patience=20, batch_size=1024, virtual_batch_size=128, eval_metric=['rmse', 'mae']):
    seed_everything(CFG.seed)

    safe_run_name = run_name.lower().replace(' ', '_').replace('/', '_')
    model_path = MODEL_DIR / f'{safe_run_name}_best'

    wandb.init(
        project=CFG.project,
        entity=CFG.entity,
        name=run_name,
        reinit=True,
        config={
            'model': 'TabNetRegressor',
            'target_transform': 'log1p(frp)',
            'n_d': n_d,
            'n_a': n_a,
            'n_steps': n_steps,
            'gamma': gamma,
            'lambda_sparse': lambda_sparse,
            'lr': lr,
            'max_epochs': max_epochs,
            'patience': patience,
            'batch_size': batch_size,
            'virtual_batch_size': virtual_batch_size,
            'eval_metric': eval_metric,
            'model_path': str(model_path)
        }
    )

    model = TabNetRegressor(
        n_d=n_d,
        n_a=n_a,
        n_steps=n_steps,
        gamma=gamma,
        lambda_sparse=lambda_sparse,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=lr),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params={'step_size': 20, 'gamma': 0.9},
        seed=CFG.seed,
        verbose=10
    )

    model.fit(
        X_train=X_train_tabnet,
        y_train=y_train_tabnet,
        eval_set=[(X_train_tabnet, y_train_tabnet), (X_val_tabnet, y_val_tabnet)],
        eval_name=['train', 'valid'],
        eval_metric=eval_metric,
        max_epochs=max_epochs,
        patience=patience,
        batch_size=batch_size,
        virtual_batch_size=virtual_batch_size,
        num_workers=0,
        drop_last=False
    )

    test_pred_log = model.predict(X_test_tabnet).reshape(-1)
    test_pred = np.expm1(test_pred_log)
    test_target = np.expm1(y_test_tabnet_log)

    metrics = regression_metrics(test_target, test_pred)
    metrics['best_epoch'] = model.best_epoch
    metrics['best_valid_score'] = model.best_cost

    saved_model_path = model.save_model(str(model_path))

    wandb.log({f'test_{key}': value for key, value in metrics.items()})
    wandb.save(saved_model_path)
    wandb.finish()

    print(f'TabNet model saved to: {saved_model_path}')
    print(f'Best epoch: {model.best_epoch}, best validation score: {model.best_cost:.4f}')

    return model, metrics


In [74]:
tabnet_model_1, tabnet_metrics_1 = train_tabnet_experiment(
    run_name='TabNetRegressor 1st Try',
    n_d=16,
    n_a=16,
    n_steps=4,
    gamma=1.3,
    lambda_sparse=1e-4,
    lr=2e-2,
    max_epochs=100,
    patience=20,
    batch_size=1024,
    virtual_batch_size=128
)

tabnet_metrics_1


C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 2.01374 | train_rmse: 0.88427 | train_mae: 0.68472 | valid_rmse: 0.88035 | valid_mae: 0.6791  |  0:00:04s
epoch 10 | loss: 0.71828 | train_rmse: 0.85352 | train_mae: 0.65515 | valid_rmse: 0.84919 | valid_mae: 0.65045 |  0:00:48s
epoch 20 | loss: 0.70946 | train_rmse: 0.83852 | train_mae: 0.64593 | valid_rmse: 0.83702 | valid_mae: 0.64322 |  0:01:30s
epoch 30 | loss: 0.70253 | train_rmse: 0.83646 | train_mae: 0.64997 | valid_rmse: 0.83441 | valid_mae: 0.64651 |  0:02:14s
epoch 40 | loss: 0.7006  | train_rmse: 0.836   | train_mae: 0.65223 | valid_rmse: 0.83304 | valid_mae: 0.64826 |  0:02:56s
epoch 50 | loss: 0.70062 | train_rmse: 0.83402 | train_mae: 0.64273 | valid_rmse: 0.83215 | valid_mae: 0.64045 |  0:03:41s
epoch 60 | loss: 0.69255 | train_rmse: 0.82933 | train_mae: 0.64319 | valid_rmse: 0.82861 | valid_mae: 0.64109 |  0:04:29s
epoch 70 | loss: 0.70051 | train_rmse: 0.83274 | train_mae: 0.64723 | valid_rmse: 0.83168 | valid_mae: 0.64589 |  0:05:12s
epoch 80 | loss:

C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Successfully saved model at models\tabnetregressor_1st_try_best.zip


test_MAE,▁
test_R2,▁
test_RMSE,▁
test_best_epoch,▁
test_best_valid_score,▁
test_MAE,12.67348
test_R2,0.0091
test_RMSE,32.60485
test_best_epoch,61
test_best_valid_score,0.63263


TabNet model saved to: models\tabnetregressor_1st_try_best.zip
Best epoch: 61, best validation score: 0.6326


{'MAE': 12.673480987548828,
 'RMSE': np.float64(32.604850773240976),
 'R2': 0.00909578800201416,
 'best_epoch': 61,
 'best_valid_score': 0.6326277852058411}

Первый TabNet остановился на 81-й эпохе, потому что max_epochs=100, patience=20, а лучшая эпоха была 61. Модель дошла до лучшей validation-метрики, подождала еще 20 эпох без улучшения и восстановила лучшие веса

Так как с baseline мы сравниваем по MAE, RMSE и R2, дальше попробуем две более компактные конфигурации. Они обучаются быстрее и проверяют, не была ли первая TabNet-модель слишком тяжелой или слишком медленно сходящейся

In [75]:
tabnet_model_2, tabnet_metrics_2 = train_tabnet_experiment(
    run_name='TabNetRegressor optimized RMSE',
    n_d=8,
    n_a=8,
    n_steps=3,
    gamma=1.1,
    lambda_sparse=1e-3,
    lr=1e-2,
    max_epochs=80,
    patience=10,
    batch_size=2048,
    virtual_batch_size=256,
    eval_metric=['rmse', 'mae']
)

tabnet_metrics_2


C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 2.25748 | train_rmse: 0.90892 | train_mae: 0.69127 | valid_rmse: 0.90654 | valid_mae: 0.69143 |  0:00:02s
epoch 10 | loss: 0.71059 | train_rmse: 0.83921 | train_mae: 0.65164 | valid_rmse: 0.83884 | valid_mae: 0.65139 |  0:00:27s
epoch 20 | loss: 0.69563 | train_rmse: 0.82977 | train_mae: 0.64554 | valid_rmse: 0.83163 | valid_mae: 0.64693 |  0:00:52s
epoch 30 | loss: 0.68835 | train_rmse: 0.82552 | train_mae: 0.64046 | valid_rmse: 0.82664 | valid_mae: 0.64059 |  0:01:18s
epoch 40 | loss: 0.68388 | train_rmse: 0.82204 | train_mae: 0.63617 | valid_rmse: 0.82473 | valid_mae: 0.6381  |  0:01:45s
epoch 50 | loss: 0.67997 | train_rmse: 0.81977 | train_mae: 0.63735 | valid_rmse: 0.82304 | valid_mae: 0.63953 |  0:02:12s

Early stopping occurred at epoch 59 with best_epoch = 49 and best_valid_mae = 0.63265


C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Successfully saved model at models\tabnetregressor_optimized_rmse_best.zip


test_MAE,▁
test_R2,▁
test_RMSE,▁
test_best_epoch,▁
test_best_valid_score,▁
test_MAE,12.6839
test_R2,0.03338
test_RMSE,32.20277
test_best_epoch,49
test_best_valid_score,0.63265


TabNet model saved to: models\tabnetregressor_optimized_rmse_best.zip
Best epoch: 49, best validation score: 0.6327


{'MAE': 12.6839017868042,
 'RMSE': np.float64(32.20277246895832),
 'R2': 0.033384501934051514,
 'best_epoch': 49,
 'best_valid_score': 0.6326522827148438}

In [76]:
tabnet_model_3, tabnet_metrics_3 = train_tabnet_experiment(
    run_name='TabNetRegressor wider MAE',
    n_d=24,
    n_a=24,
    n_steps=3,
    gamma=1.2,
    lambda_sparse=1e-4,
    lr=5e-3,
    max_epochs=80,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128,
    eval_metric=['mae', 'rmse']
)

tabnet_metrics_3


C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.23384 | train_mae: 0.68661 | train_rmse: 0.88637 | valid_mae: 0.68449 | valid_rmse: 0.88468 |  0:00:04s
epoch 10 | loss: 0.70637 | train_mae: 0.65078 | train_rmse: 0.83499 | valid_mae: 0.64984 | valid_rmse: 0.83505 |  0:00:53s
epoch 20 | loss: 0.69565 | train_mae: 0.63843 | train_rmse: 0.82989 | valid_mae: 0.6384  | valid_rmse: 0.8315  |  0:01:32s
epoch 30 | loss: 0.689   | train_mae: 0.64634 | train_rmse: 0.82823 | valid_mae: 0.64826 | valid_rmse: 0.83148 |  0:02:11s
epoch 40 | loss: 0.68181 | train_mae: 0.63572 | train_rmse: 0.82083 | valid_mae: 0.63786 | valid_rmse: 0.82415 |  0:02:56s
epoch 50 | loss: 0.67982 | train_mae: 0.63405 | train_rmse: 0.81889 | valid_mae: 0.63652 | valid_rmse: 0.82393 |  0:03:37s

Early stopping occurred at epoch 57 with best_epoch = 47 and best_valid_rmse = 0.82328


C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Successfully saved model at models\tabnetregressor_wider_mae_best.zip


test_MAE,▁
test_R2,▁
test_RMSE,▁
test_best_epoch,▁
test_best_valid_score,▁
test_MAE,12.72997
test_R2,0.037
test_RMSE,32.14255
test_best_epoch,47
test_best_valid_score,0.82328


TabNet model saved to: models\tabnetregressor_wider_mae_best.zip
Best epoch: 47, best validation score: 0.8233


{'MAE': 12.729969024658203,
 'RMSE': np.float64(32.14254673399223),
 'R2': 0.036996662616729736,
 'best_epoch': 47,
 'best_valid_score': np.float64(0.8232767588821791)}

### Вывод по TabNet-моделям

По результатам экспериментов TabNet не решил основную проблему и не смог обогнать baseline RandomForest

TabNet обучается корректно, использует early stopping и восстанавливает лучшие веса, но качество на тестовых данных остается хуже baseline. Более компактная конфигурация улучшила RMSE и R2 относительно первого TabNet, но этого оказалось недостаточно, чтобы превзойти RandomForest

## 6. Нейросеть + учитель

После MLP и TabNet стало видно, что нейросетевые модели плохо обгоняют baseline, если обучать их напрямую предсказывать frp_log. Поэтому сейчас попробуем другую идею - не заставлять нейросеть решать всю задачу с нуля, а дать ей более узкую задачу - исправлять ошибки сильной табличной модели.

### Основная идея 

Метод называется Residual learning и означает обучение на остатках, то есть на ошибках уже существующей модели.

Пусть базовая модель строит прогноз:

teacher_prediction = teacher_model(X)

Тогда ошибка базовой модели:

residual = true_frp_log - teacher_prediction

Нейросеть обучается не предсказывать саму целевую переменную, а предсказывать этот residual:

residual_mlp_prediction = MLP(X, teacher_prediction)

Финальный прогноз строится так:

final_prediction = teacher_prediction + alpha * residual_mlp_prediction


alpha - коэффициент, который показывает, насколько сильно мы доверяем исправлению нейросети.

### Отличия от предыдущих моделей

Обычная MLP должна сразу выучить всю зависимость между признаками и frp. Для табличных данных это не самая простая задача. В residual ensemble большая часть работы перекладывается на сильную модель учитель. В нашем случае мы выбрали ExtraTreesRegressor. Она хорошо работает с табличными данными, нелинейностями и взаимодействиями признаков. После этого MLP решает более простую задачу: найти систематические ошибки teacher-модели и скорректировать их.

То есть нейросеть используется не как замена деревьям, а как дополнительный слой поверх сильной табличной модели.

### Почему используется ExtraTreesRegressor

ExtraTreesRegressor похож на RandomForest, который есть у нас в бейзлайн модели, но добавляет больше случайности при построении деревьев. За счет этого он часто снижает variance и может лучше обобщаться на тестовой выборке

Для нашей задачи это полезно, потому что уелевая переменная шумная, а модель должна быть устойчивой к выбросам и случайным колебаниям в данных.

Residual MLP получает не только исходные признаки, но и прогноз teacher-модели. Прогноз модели является сильным агрегированным признаком. Он уже содержит информацию, которую деревья извлекли из исходных данных. MLP может использовать этот прогноз как основу и учиться, когда его нужно увеличить или уменьшить

Если просто прибавить весь residual-прогноз нейросети, можно переусилить коррекцию и ухудшить качество. Поэтому коэффициент alpha подбирается на validation set. По факту это гиперпараметр

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
#Создаем нейросеть со слями [256, 128, 64] и SiLU активацией, а также с BatchNorm и Dropout для лучшей стабильности обучения.
class ResidualCorrectionNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.05),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(0.05),
            nn.Linear(128, 64),
            nn.SiLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)


def train_neural_residual_ensemble(run_name='ExtraTrees teacher + residual MLP'):
    seed_everything(CFG.seed)
    #Создаем дерево учитель с параметрами, которые показались оптимальными при ручной настройке на валидации.
    teacher = ExtraTreesRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        random_state=CFG.seed,
        n_jobs=-1
    )
    #Обучаем модель учителя
    teacher.fit(X_train_scaled, y_train_log)

    teacher_train_pred = teacher.predict(X_train_scaled).astype('float32')
    teacher_val_pred = teacher.predict(X_val_scaled).astype('float32')
    teacher_test_pred = teacher.predict(X_test_scaled).astype('float32')

    X_train_stack = np.column_stack([X_train_scaled, teacher_train_pred]).astype('float32')
    X_val_stack = np.column_stack([X_val_scaled, teacher_val_pred]).astype('float32')
    X_test_stack = np.column_stack([X_test_scaled, teacher_test_pred]).astype('float32')

    residual_train = (y_train_log - teacher_train_pred).astype('float32')
    residual_val = (y_val_log - teacher_val_pred).astype('float32')

    train_residual_loader = DataLoader(
        TensorDataset(
            torch.FloatTensor(X_train_stack),
            torch.FloatTensor(residual_train).view(-1, 1)
        ),
        batch_size=512,
        shuffle=True
    )
    #Создаем нейросеть с функцией потерь HuberLoss и оптимизатором AdamW
    model = ResidualCorrectionNet(input_size=X_train_stack.shape[1]).to(DEVICE)
    criterion = nn.HuberLoss(delta=0.3) 
    optimizer = optim.AdamW(model.parameters(), lr=7e-4, weight_decay=1e-4)

    teacher_path = MODEL_DIR / 'extratrees_teacher_best.joblib'
    residual_model_path = MODEL_DIR / 'extratrees_residual_mlp_best.pt'

    wandb.init(
        project=CFG.project,
        entity=CFG.entity,
        name=run_name,
        reinit=True,
        config={
            'model': run_name,
            'teacher': 'ExtraTreesRegressor',
            'residual_model': 'ResidualCorrectionNet',
            'n_estimators': 500,
            'min_samples_leaf': 2,
            'loss': 'HuberLoss(delta=0.3)',
            'lr': 7e-4,
            'weight_decay': 1e-4,
            'target_transform': 'log1p(frp)'
        }
    )

    best_val_loss = np.inf
    best_epoch = 0
    best_state = copy.deepcopy(model.state_dict())
    patience = 0

    X_val_stack_tensor = torch.FloatTensor(X_val_stack).to(DEVICE)
    residual_val_tensor = torch.FloatTensor(residual_val).view(-1, 1).to(DEVICE)
    #Обучать будем 120 эпох с ранней остановкой если качество не улучшается 20 эпох подряд
    for epoch in tqdm(range(1, 121), desc=run_name):
        model.train()
        train_losses = []

        for batch_x, batch_y in train_residual_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_stack_tensor), residual_val_tensor).item()

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            torch.save({
                'model_state_dict': best_state,
                'input_size': X_train_stack.shape[1],
                'best_epoch': best_epoch,
                'best_val_loss': best_val_loss,
                'alpha': None,
                'target_transform': 'log1p(frp)'
            }, residual_model_path)
            patience = 0
        else:
            patience += 1

        wandb.log({
            'epoch': epoch,
            'train_residual_loss': float(np.mean(train_losses)),
            'val_residual_loss': val_loss,
            'best_val_residual_loss': best_val_loss
        })

        if epoch % 10 == 0:
            print(f'Epoch: {epoch}, val_residual_loss: {val_loss:.4f}, best: {best_val_loss:.4f}')

        if patience >= 20:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        residual_val_pred = model(torch.FloatTensor(X_val_stack).to(DEVICE)).cpu().numpy().reshape(-1)
        residual_test_pred = model(torch.FloatTensor(X_test_stack).to(DEVICE)).cpu().numpy().reshape(-1)

    best_alpha = 0.0
    best_val_mae = np.inf

    for alpha in np.linspace(-0.5, 0.5, 81):
        val_pred_log = teacher_val_pred + alpha * residual_val_pred
        val_pred = np.expm1(val_pred_log)
        val_target = np.expm1(y_val_log)
        val_mae = mean_absolute_error(val_target, val_pred)

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_alpha = float(alpha)

    test_pred_log = teacher_test_pred + best_alpha * residual_test_pred
    test_pred = np.expm1(test_pred_log)
    test_target = np.expm1(y_test_log)

    metrics = regression_metrics(test_target, test_pred)
    metrics['best_epoch'] = best_epoch
    metrics['best_val_loss'] = best_val_loss
    metrics['alpha'] = best_alpha

    joblib.dump(teacher, teacher_path)
    torch.save({
        'model_state_dict': best_state,
        'input_size': X_train_stack.shape[1],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'alpha': best_alpha,
        'target_transform': 'log1p(frp)'
    }, residual_model_path)

    wandb.log({f'test_{key}': value for key, value in metrics.items()})
    wandb.save(str(teacher_path))
    wandb.save(str(residual_model_path))
    wandb.finish()

    print(f'Teacher model saved to: {teacher_path}')
    print(f'Residual MLP saved to: {residual_model_path}')
    print(f'Best epoch: {best_epoch}, alpha: {best_alpha}')

    return teacher, model, metrics


ensemble_teacher, ensemble_residual_model, ensemble_metrics = train_neural_residual_ensemble()
ensemble_metrics


ExtraTrees teacher + residual MLP:   0%|          | 0/120 [00:00<?, ?it/s]

Epoch: 10, val_residual_loss: 0.1462, best: 0.1459
Epoch: 20, val_residual_loss: 0.1462, best: 0.1459


wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.
wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


best_val_residual_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
test_MAE,▁
test_R2,▁
test_RMSE,▁
test_alpha,▁
test_best_epoch,▁
test_best_val_loss,▁
train_residual_loss,█▅▄▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val_residual_loss,▅▁█▅▂▄▂▁▂▃▄▃▂▅▁▅▃▄▃▃▅▅
best_val_residual_loss,0.14593


Teacher model saved to: models\extratrees_teacher_best.joblib
Residual MLP saved to: models\extratrees_residual_mlp_best.pt
Best epoch: 2, alpha: -0.5


{'MAE': 12.461560249328613,
 'RMSE': np.float64(30.707896615541436),
 'R2': 0.12104350328445435,
 'best_epoch': 2,
 'best_val_loss': 0.14592525362968445,
 'alpha': -0.5}